# 第19章 计算思维与问题求解（一）：递归与分治
# Chapter 19 Computational Thinking & Problem Solving (1): Recursion & Divide and Conquer

---

## Cambridge A-Level Computer Science (9618) - Paper 4

**本节学习目标 Learning Objectives:**

| 编号 | 目标 | 英文 |
|------|------|------|
| 1 | 理解递归的概念：基线条件与递归条件 | Understand recursion: base case and recursive case |
| 2 | 能够追踪递归调用栈（call stack） | Trace through recursive call stacks |
| 3 | 掌握汉诺塔问题的递归解法 | Solve Tower of Hanoi recursively |
| 4 | 理解分治策略（Divide and Conquer） | Understand Divide and Conquer strategy |
| 5 | 实现并分析归并排序（Merge Sort） | Implement and analyze Merge Sort |
| 6 | 实现并分析快速排序（Quick Sort） | Implement and analyze Quick Sort |
| 7 | 实现递归版二分查找（Binary Search） | Implement recursive Binary Search |
| 8 | 比较 Fibonacci 的不同实现方法 | Compare different Fibonacci implementations |

In [ ]:
# 环境配置 - Environment Setup
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import time
import sys
from functools import lru_cache

# 中文字体配置 - Chinese font configuration
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("环境配置完成！Ready to go!")

---
## 1. 什么是递归？What is Recursion?

**递归（Recursion）** 是一种编程技巧，指的是 **函数调用自身** 来解决问题。

### 递归的两个核心要素：

| 要素 | 英文 | 说明 |
|------|------|------|
| **基线条件** | Base Case | 递归停止的条件，防止无限递归 |
| **递归条件** | Recursive Case | 函数调用自身，问题规模缩小 |

### 类比理解：
想象你站在一排镜子前面，镜子里面还有镜子，一层层嵌套下去。
但是最终一定要有一个 **终止点**（base case），否则就会无穷无尽（无限递归 → Stack Overflow）。

```
递归函数结构：
def recursive_function(参数):
    if 基线条件:          # Base case
        return 基本值
    else:                  # Recursive case
        return recursive_function(更小的参数)
```

In [ ]:
# 最简单的递归示例：计算阶乘 Factorial
# n! = n × (n-1) × (n-2) × ... × 1
# 例如: 5! = 5 × 4 × 3 × 2 × 1 = 120

def factorial(n):
    """
    递归计算阶乘
    Recursively calculate factorial
    
    Base case:  0! = 1 或 1! = 1
    Recursive:  n! = n × (n-1)!
    """
    # 基线条件 Base case
    if n == 0 or n == 1:
        print(f"  基线条件 Base case: factorial({n}) = 1")
        return 1
    # 递归条件 Recursive case
    else:
        print(f"  递归调用 Recursive call: factorial({n}) = {n} × factorial({n-1})")
        result = n * factorial(n - 1)
        print(f"  返回 Return: factorial({n}) = {result}")
        return result

print("=== 计算 5! ===")
print(f"\n最终结果 Final result: 5! = {factorial(5)}")

---
## 2. 调用栈可视化 Call Stack Visualization

当递归函数被调用时，每一次调用都会在 **调用栈（Call Stack）** 中创建一个新的 **栈帧（Stack Frame）**。

调用栈的工作方式：
1. 每次函数调用 → 压入栈（push）
2. 函数返回 → 弹出栈（pop）
3. 最先调用的最后返回（LIFO - Last In First Out）

In [ ]:
# 可视化调用栈 - Visualize the Call Stack for factorial(5)

def visualize_call_stack():
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    
    # 左图：递归展开过程 (Unwinding)
    ax1 = axes[0]
    ax1.set_title('递归展开（压栈过程）\nRecursive Calls (Push)', fontsize=14, fontweight='bold')
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 12)
    ax1.axis('off')
    
    calls = [
        ('factorial(5)', '5 × factorial(4)', '#FF6B6B'),
        ('factorial(4)', '4 × factorial(3)', '#FFA07A'),
        ('factorial(3)', '3 × factorial(2)', '#FFD700'),
        ('factorial(2)', '2 × factorial(1)', '#90EE90'),
        ('factorial(1)', 'return 1 (基线条件)', '#87CEEB'),
    ]
    
    for i, (func, detail, color) in enumerate(calls):
        y = 10 - i * 2
        rect = mpatches.FancyBboxPatch((1, y - 0.6), 8, 1.2, 
                                        boxstyle="round,pad=0.1", 
                                        facecolor=color, edgecolor='black', linewidth=2)
        ax1.add_patch(rect)
        ax1.text(5, y + 0.15, func, ha='center', va='center', fontsize=11, fontweight='bold')
        ax1.text(5, y - 0.25, detail, ha='center', va='center', fontsize=9)
        if i < len(calls) - 1:
            ax1.annotate('', xy=(5, y - 0.7), xytext=(5, y - 1.3),
                        arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    ax1.text(0.5, 11, '调用方向 →', fontsize=10, color='red')
    
    # 右图：返回过程 (Rewinding)
    ax2 = axes[1]
    ax2.set_title('递归返回（弹栈过程）\nReturn Values (Pop)', fontsize=14, fontweight='bold')
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 12)
    ax2.axis('off')
    
    returns = [
        ('factorial(1) = 1', '#87CEEB'),
        ('factorial(2) = 2 × 1 = 2', '#90EE90'),
        ('factorial(3) = 3 × 2 = 6', '#FFD700'),
        ('factorial(4) = 4 × 6 = 24', '#FFA07A'),
        ('factorial(5) = 5 × 24 = 120', '#FF6B6B'),
    ]
    
    for i, (text, color) in enumerate(returns):
        y = 2 + i * 2
        rect = mpatches.FancyBboxPatch((1, y - 0.5), 8, 1.0,
                                        boxstyle="round,pad=0.1",
                                        facecolor=color, edgecolor='black', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(5, y, text, ha='center', va='center', fontsize=10, fontweight='bold')
        if i < len(returns) - 1:
            ax2.annotate('', xy=(5, y + 0.6), xytext=(5, y + 1.3),
                        arrowprops=dict(arrowstyle='->', color='green', lw=2))
    
    ax2.text(0.5, 11, '← 返回方向', fontsize=10, color='green')
    
    plt.tight_layout()
    plt.savefig('call_stack_visualization.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_call_stack()

---
## 3. 汉诺塔 Tower of Hanoi

汉诺塔是经典的递归问题：

**规则：**
- 有三根柱子 A、B、C
- A 柱上有 n 个大小不同的圆盘，从上到下由小到大排列
- 目标：把所有圆盘从 A 移到 C
- 限制：每次只能移动一个盘子，且大盘子不能放在小盘子上面

**递归思路：**
1. 把上面 n-1 个盘子从 A 移到 B（借助 C）
2. 把最大的盘子从 A 移到 C
3. 把 n-1 个盘子从 B 移到 C（借助 A）

**移动次数公式：** $T(n) = 2^n - 1$

In [ ]:
# 汉诺塔 - Tower of Hanoi with step-by-step trace

move_count = 0

def hanoi(n, source, target, auxiliary, depth=0):
    """
    汉诺塔递归解法
    n: 盘子数量
    source: 源柱子
    target: 目标柱子
    auxiliary: 辅助柱子
    depth: 递归深度（用于缩进显示）
    """
    global move_count
    indent = "  " * depth
    
    if n == 1:
        # 基线条件：只有一个盘子，直接移动
        move_count += 1
        print(f"{indent}步骤 {move_count}: 移动盘子 1 从 {source} → {target}")
        return
    
    # 第1步：将 n-1 个盘子从 source 移到 auxiliary
    print(f"{indent}将 {n-1} 个盘子从 {source} 移到 {auxiliary}（借助 {target}）:")
    hanoi(n - 1, source, auxiliary, target, depth + 1)
    
    # 第2步：将最大的盘子从 source 移到 target
    move_count += 1
    print(f"{indent}步骤 {move_count}: 移动盘子 {n} 从 {source} → {target} ★")
    
    # 第3步：将 n-1 个盘子从 auxiliary 移到 target
    print(f"{indent}将 {n-1} 个盘子从 {auxiliary} 移到 {target}（借助 {source}）:")
    hanoi(n - 1, auxiliary, target, source, depth + 1)

print("=== 汉诺塔 3 个盘子 ===")
print("从柱子 A 移到柱子 C，借助柱子 B\n")
move_count = 0
hanoi(3, 'A', 'C', 'B')
print(f"\n总移动次数: {move_count} (公式: 2^3 - 1 = {2**3 - 1})")

In [ ]:
# 汉诺塔可视化 - Tower of Hanoi Visualization

def visualize_hanoi_steps(n=3):
    """可视化汉诺塔的移动过程"""
    # 记录所有步骤
    steps = []
    pegs = {'A': list(range(n, 0, -1)), 'B': [], 'C': []}
    
    # 保存初始状态
    steps.append({k: v[:] for k, v in pegs.items()})
    
    def hanoi_record(num, src, dst, aux):
        if num == 1:
            disk = pegs[src].pop()
            pegs[dst].append(disk)
            steps.append({k: v[:] for k, v in pegs.items()})
            return
        hanoi_record(num - 1, src, aux, dst)
        disk = pegs[src].pop()
        pegs[dst].append(disk)
        steps.append({k: v[:] for k, v in pegs.items()})
        hanoi_record(num - 1, aux, dst, src)
    
    hanoi_record(n, 'A', 'C', 'B')
    
    # 选取关键步骤展示
    show_indices = [0, len(steps)//4, len(steps)//2, 3*len(steps)//4, len(steps)-1]
    show_indices = sorted(set(show_indices))
    
    fig, axes = plt.subplots(1, len(show_indices), figsize=(4*len(show_indices), 5))
    if len(show_indices) == 1:
        axes = [axes]
    colors = ['#FF6B6B', '#FFD700', '#87CEEB', '#90EE90', '#DDA0DD']
    
    for idx, (ax, step_i) in enumerate(zip(axes, show_indices)):
        state = steps[step_i]
        ax.set_xlim(-0.5, 3.5)
        ax.set_ylim(-0.5, n + 1.5)
        ax.set_title(f'步骤 {step_i}', fontsize=12, fontweight='bold')
        ax.axis('off')
        
        for j, peg_name in enumerate(['A', 'B', 'C']):
            x = j + 0.5
            # 画柱子
            ax.plot([x, x], [0, n + 0.5], 'k-', linewidth=3)
            ax.text(x, -0.3, peg_name, ha='center', fontsize=12, fontweight='bold')
            # 画底座
            ax.plot([x - 0.4, x + 0.4], [0, 0], 'k-', linewidth=4)
            # 画盘子
            for k, disk in enumerate(state[peg_name]):
                width = 0.15 + disk * 0.2 / n
                rect = mpatches.FancyBboxPatch((x - width/2, k * 0.4 + 0.1), width, 0.35,
                                                boxstyle="round,pad=0.02",
                                                facecolor=colors[disk-1], edgecolor='black')
                ax.add_patch(rect)
                ax.text(x, k * 0.4 + 0.27, str(disk), ha='center', va='center', fontsize=8)
    
    plt.suptitle(f'汉诺塔 Tower of Hanoi (n={n}) 关键步骤', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_hanoi_steps(3)

---
## 4. 分治策略 Divide and Conquer Strategy

**分治法（Divide and Conquer）** 是一种重要的算法设计范式：

### 三个步骤：

| 步骤 | 英文 | 说明 |
|------|------|------|
| **分解** | Divide | 将原问题分解为若干个规模较小的子问题 |
| **解决** | Conquer | 递归地解决各个子问题（当子问题足够小时直接求解） |
| **合并** | Combine | 将子问题的解合并为原问题的解 |

```
         原问题 (Original Problem)
        /         |
    子问题1      子问题2       ← Divide
    /    \       /    \
  子子1  子子2  子子3  子子4   ← Conquer (递归)
    \    /       \    /
    结果1        结果2         ← Combine
        \         /
        最终结果
```

### 经典分治算法：
- **归并排序 Merge Sort**
- **快速排序 Quick Sort**
- **二分查找 Binary Search**

---
## 5. 归并排序 Merge Sort

归并排序是分治法的经典应用：

1. **Divide（分）:** 将数组从中间分成两半
2. **Conquer（治）:** 递归地对两半分别排序
3. **Combine（合）:** 合并两个已排序的子数组

**时间复杂度：** O(n log n) — 所有情况下都是这个复杂度

**空间复杂度：** O(n) — 需要额外空间存储临时数组

In [ ]:
# 归并排序 - Merge Sort with detailed trace

def merge_sort(arr, depth=0):
    """
    归并排序（含详细过程追踪）
    Merge Sort with step-by-step trace
    """
    indent = "  " * depth
    print(f"{indent}merge_sort({arr})")
    
    # 基线条件：数组长度 <= 1，已经有序
    if len(arr) <= 1:
        print(f"{indent}  → 基线条件，返回 {arr}")
        return arr
    
    # Divide: 从中间分割
    mid = len(arr) // 2
    left = arr[:mid]
    right = arr[mid:]
    print(f"{indent}  分割 Divide: 左={left}, 右={right}")
    
    # Conquer: 递归排序两半
    sorted_left = merge_sort(left, depth + 1)
    sorted_right = merge_sort(right, depth + 1)
    
    # Combine: 合并两个有序数组
    merged = merge(sorted_left, sorted_right)
    print(f"{indent}  合并 Merge: {sorted_left} + {sorted_right} → {merged}")
    return merged


def merge(left, right):
    """合并两个有序数组 Merge two sorted arrays"""
    result = []
    i = j = 0
    
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    
    # 添加剩余元素
    result.extend(left[i:])
    result.extend(right[j:])
    return result


print("=== 归并排序详细过程 ===")
data = [38, 27, 43, 3, 9, 82, 10]
print(f"原始数组: {data}\n")
result = merge_sort(data)
print(f"\n排序结果: {result}")

In [ ]:
# 归并排序可视化 - Merge Sort Visualization

def visualize_merge_sort():
    """可视化归并排序的分割和合并过程"""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('归并排序过程可视化 Merge Sort Process', fontsize=16, fontweight='bold')
    
    # 定义各层的数据
    levels = [
        # (y, 数据列表, x起始位置列表)
        (9, ['[38, 27, 43, 3, 9, 82, 10]'], [8]),
        (7.5, ['[38, 27, 43]', '[3, 9, 82, 10]'], [4.5, 11.5]),
        (6, ['[38]', '[27, 43]', '[3, 9]', '[82, 10]'], [2.5, 6, 10, 13.5]),
        (4.5, ['[38]', '[27]', '[43]', '[3]', '[9]', '[82]', '[10]'], [2.5, 5, 7, 9.5, 10.5, 12.5, 14.5]),
    ]
    
    merge_levels = [
        (3, ['[38]', '[27, 43]', '[3, 9]', '[10, 82]'], [2.5, 6, 10, 13.5]),
        (1.5, ['[27, 38, 43]', '[3, 9, 10, 82]'], [4.5, 11.5]),
        (0, ['[3, 9, 10, 27, 38, 43, 82]'], [8]),
    ]
    
    # 分割阶段标记
    ax.text(0.3, 9, '分割\nDivide\n  ↓', fontsize=10, color='red', fontweight='bold', va='top')
    ax.text(0.3, 3, '合并\nMerge\n  ↑', fontsize=10, color='green', fontweight='bold', va='top')
    
    colors_divide = ['#FFB3BA', '#FFDFBA', '#FFFFBA', '#BAFFC9']
    colors_merge = ['#BAFFC9', '#FFFFBA', '#BAE1FF']
    
    for level_idx, (y, items, positions) in enumerate(levels):
        for item, x in zip(items, positions):
            w = len(item) * 0.18 + 0.5
            rect = mpatches.FancyBboxPatch((x - w/2, y - 0.3), w, 0.6,
                                            boxstyle="round,pad=0.05",
                                            facecolor=colors_divide[level_idx], edgecolor='red', linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x, y, item, ha='center', va='center', fontsize=9, fontweight='bold')
    
    for level_idx, (y, items, positions) in enumerate(merge_levels):
        for item, x in zip(items, positions):
            w = len(item) * 0.18 + 0.5
            rect = mpatches.FancyBboxPatch((x - w/2, y - 0.3), w, 0.6,
                                            boxstyle="round,pad=0.05",
                                            facecolor=colors_merge[level_idx], edgecolor='green', linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x, y, item, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # 分隔线
    ax.axhline(y=3.7, color='gray', linestyle='--', linewidth=1)
    ax.text(8, 3.85, '--- 分割完成，开始合并 ---', ha='center', fontsize=10, color='gray')
    
    plt.tight_layout()
    plt.show()

visualize_merge_sort()

---
## 6. 快速排序 Quick Sort

快速排序也是分治法的应用，但策略不同：

1. **选择基准值（Pivot）:** 从数组中选一个元素作为基准
2. **分区（Partition）:** 将数组分为三部分：
   - 小于 pivot 的元素
   - 等于 pivot 的元素
   - 大于 pivot 的元素
3. **递归排序** 左右两部分

**时间复杂度：**
- 最好/平均：O(n log n)
- 最坏：O(n^2)（当数组已排序且总选第一个元素作 pivot 时）

**空间复杂度：** O(log n)（递归调用栈）

In [ ]:
# 快速排序 - Quick Sort with detailed trace

def quick_sort(arr, depth=0):
    """
    快速排序（含详细过程追踪）
    Quick Sort with step-by-step trace
    """
    indent = "  " * depth
    print(f"{indent}quick_sort({arr})")
    
    # 基线条件
    if len(arr) <= 1:
        print(f"{indent}  → 基线条件，返回 {arr}")
        return arr
    
    # 选择 pivot（这里选中间元素）
    pivot = arr[len(arr) // 2]
    print(f"{indent}  Pivot = {pivot}")
    
    # 分区 Partition
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    print(f"{indent}  分区: 左={left}, 中={middle}, 右={right}")
    
    # 递归排序并合并
    sorted_left = quick_sort(left, depth + 1)
    sorted_right = quick_sort(right, depth + 1)
    
    result = sorted_left + middle + sorted_right
    print(f"{indent}  结果: {result}")
    return result


print("=== 快速排序详细过程 ===")
data = [38, 27, 43, 3, 9, 82, 10]
print(f"原始数组: {data}\n")
result = quick_sort(data)
print(f"\n排序结果: {result}")

In [ ]:
# 原地快速排序（考试常考版本）- In-place Quick Sort

def quick_sort_inplace(arr, low, high):
    """
    原地快速排序（不需要额外空间）
    In-place Quick Sort (exam-style implementation)
    """
    if low < high:
        # 分区并获取 pivot 的最终位置
        pivot_index = partition(arr, low, high)
        # 递归排序左右两部分
        quick_sort_inplace(arr, low, pivot_index - 1)
        quick_sort_inplace(arr, pivot_index + 1, high)


def partition(arr, low, high):
    """
    Lomuto 分区方案
    选最后一个元素作为 pivot
    """
    pivot = arr[high]  # 选最后一个元素作为基准
    i = low - 1        # i 指向小于 pivot 区域的最后一个元素
    
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]  # 交换
    
    # 把 pivot 放到正确位置
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1


# 测试
data = [38, 27, 43, 3, 9, 82, 10]
print(f"原始数组: {data}")
quick_sort_inplace(data, 0, len(data) - 1)
print(f"排序结果: {data}")
print("\n注意：原地排序直接修改原数组，不需要额外空间！")

In [ ]:
# 排序过程动画可视化 - Sorting Steps Visualization

def visualize_sorting_comparison():
    """可视化归并排序 vs 快速排序的比较次数"""
    import random
    
    # 记录比较次数
    merge_comparisons = [0]
    quick_comparisons = [0]
    
    def merge_sort_count(arr):
        if len(arr) <= 1:
            return arr
        mid = len(arr) // 2
        left = merge_sort_count(arr[:mid])
        right = merge_sort_count(arr[mid:])
        result = []
        i = j = 0
        while i < len(left) and j < len(right):
            merge_comparisons[0] += 1
            if left[i] <= right[j]:
                result.append(left[i])
                i += 1
            else:
                result.append(right[j])
                j += 1
        result.extend(left[i:])
        result.extend(right[j:])
        return result
    
    def quick_sort_count(arr):
        if len(arr) <= 1:
            return arr
        pivot = arr[len(arr) // 2]
        left, middle, right = [], [], []
        for x in arr:
            quick_comparisons[0] += 1
            if x < pivot:
                left.append(x)
            elif x == pivot:
                middle.append(x)
            else:
                right.append(x)
        return quick_sort_count(left) + middle + quick_sort_count(right)
    
    sizes = [10, 50, 100, 200, 500, 1000]
    merge_counts = []
    quick_counts = []
    
    for size in sizes:
        data = list(range(size))
        random.shuffle(data)
        data2 = data[:]
        
        merge_comparisons[0] = 0
        merge_sort_count(data)
        merge_counts.append(merge_comparisons[0])
        
        quick_comparisons[0] = 0
        quick_sort_count(data2)
        quick_counts.append(quick_comparisons[0])
    
    # 绘图
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(sizes))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, merge_counts, width, label='归并排序 Merge Sort', color='#4ECDC4')
    bars2 = ax.bar(x + width/2, quick_counts, width, label='快速排序 Quick Sort', color='#FF6B6B')
    
    # n*log(n) 参考线
    nlogn = [s * np.log2(s) for s in sizes]
    ax.plot(x, nlogn, 'g--', linewidth=2, label='n log n (理论值)', marker='o')
    
    ax.set_xlabel('数组大小 Array Size', fontsize=12)
    ax.set_ylabel('比较次数 Number of Comparisons', fontsize=12)
    ax.set_title('归并排序 vs 快速排序：比较次数对比', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(sizes)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_sorting_comparison()

---
## 7. 递归版二分查找 Recursive Binary Search

二分查找的前提条件：**数组必须已排序**

### 分治思路：
1. **Divide:** 取中间元素与目标值比较
2. **Conquer:** 
   - 如果相等 → 找到了（基线条件）
   - 如果目标 < 中间 → 在左半部分递归查找
   - 如果目标 > 中间 → 在右半部分递归查找
3. 如果 low > high → 没找到（基线条件）

**时间复杂度：** O(log n)

In [ ]:
# 递归二分查找 - Recursive Binary Search

def binary_search(arr, target, low, high, depth=0):
    """
    递归二分查找
    Recursive Binary Search
    
    返回目标元素的索引，未找到返回 -1
    """
    indent = "  " * depth
    print(f"{indent}搜索范围: arr[{low}:{high+1}] = {arr[low:high+1]}")
    
    # 基线条件1：范围无效，未找到
    if low > high:
        print(f"{indent}  → 未找到！low > high")
        return -1
    
    mid = (low + high) // 2
    print(f"{indent}  中间索引 mid={mid}, arr[{mid}]={arr[mid]}, 目标={target}")
    
    # 基线条件2：找到目标
    if arr[mid] == target:
        print(f"{indent}  → 找到了！位置 = {mid}")
        return mid
    # 递归条件：在左半部分查找
    elif target < arr[mid]:
        print(f"{indent}  → {target} < {arr[mid]}，搜索左半部分")
        return binary_search(arr, target, low, mid - 1, depth + 1)
    # 递归条件：在右半部分查找
    else:
        print(f"{indent}  → {target} > {arr[mid]}，搜索右半部分")
        return binary_search(arr, target, mid + 1, high, depth + 1)


# 测试
sorted_arr = [3, 9, 10, 27, 38, 43, 82]
print(f"有序数组: {sorted_arr}")
print(f"索引:     {list(range(len(sorted_arr)))}")

print("\n=== 查找 27 ===")
result = binary_search(sorted_arr, 27, 0, len(sorted_arr) - 1)

print("\n=== 查找 50（不存在）===")
result = binary_search(sorted_arr, 50, 0, len(sorted_arr) - 1)

In [ ]:
# 二分查找可视化 - Binary Search Visualization

def visualize_binary_search(arr, target):
    """可视化二分查找的每一步"""
    steps = []
    
    def bs(low, high):
        if low > high:
            return -1
        mid = (low + high) // 2
        steps.append((low, mid, high))
        if arr[mid] == target:
            return mid
        elif target < arr[mid]:
            return bs(low, mid - 1)
        else:
            return bs(mid + 1, high)
    
    result = bs(0, len(arr) - 1)
    
    fig, axes = plt.subplots(len(steps), 1, figsize=(12, 2.5 * len(steps)))
    if len(steps) == 1:
        axes = [axes]
    
    for step_idx, (ax, (low, mid, high)) in enumerate(zip(axes, steps)):
        ax.set_xlim(-0.5, len(arr) - 0.5)
        ax.set_ylim(-0.5, 1.5)
        ax.set_title(f'步骤 {step_idx + 1}: low={low}, mid={mid}, high={high}', fontsize=12)
        ax.axis('off')
        
        for i, val in enumerate(arr):
            if i == mid:
                color = '#FFD700' if arr[mid] != target else '#90EE90'
                lw = 3
            elif low <= i <= high:
                color = '#87CEEB'
                lw = 1.5
            else:
                color = '#E0E0E0'
                lw = 1
            
            rect = mpatches.FancyBboxPatch((i - 0.4, 0.1), 0.8, 0.8,
                                            boxstyle="round,pad=0.05",
                                            facecolor=color, edgecolor='black', linewidth=lw)
            ax.add_patch(rect)
            ax.text(i, 0.5, str(val), ha='center', va='center', fontsize=14, fontweight='bold')
            ax.text(i, -0.15, str(i), ha='center', va='center', fontsize=9, color='gray')
        
        # 标记 mid
        ax.annotate('mid', xy=(mid, 1.0), xytext=(mid, 1.3),
                   fontsize=10, ha='center', color='red', fontweight='bold',
                   arrowprops=dict(arrowstyle='->', color='red'))
    
    found_text = f'找到 {target} 在索引 {result}' if result != -1 else f'{target} 未找到'
    plt.suptitle(f'二分查找: 目标={target} → {found_text}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_binary_search([3, 9, 10, 27, 38, 43, 82], 27)

---
## 8. Fibonacci 数列：三种实现方法对比

Fibonacci 数列：0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ...

定义：
- F(0) = 0
- F(1) = 1  
- F(n) = F(n-1) + F(n-2)，当 n ≥ 2

### 三种方法对比：

| 方法 | 时间复杂度 | 空间复杂度 | 说明 |
|------|-----------|-----------|------|
| 朴素递归 Naive Recursion | O(2^n) | O(n) | 大量重复计算 |
| 记忆化 Memoization | O(n) | O(n) | 缓存已计算结果 |
| 动态规划 Dynamic Programming | O(n) | O(n) 或 O(1) | 自底向上计算 |

In [ ]:
# 方法一：朴素递归（效率极低）
# Method 1: Naive Recursion (very slow)

call_count_naive = 0

def fib_naive(n):
    """朴素递归 - O(2^n) 时间复杂度"""
    global call_count_naive
    call_count_naive += 1
    
    if n <= 1:  # Base case
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)  # Recursive case


# 测试小数字
for n in [5, 10, 20, 30]:
    call_count_naive = 0
    start = time.time()
    result = fib_naive(n)
    elapsed = time.time() - start
    print(f"fib({n:2d}) = {result:>10d} | 调用次数: {call_count_naive:>12,d} | 耗时: {elapsed:.4f}秒")

print("\n注意观察：n 每增加一点，调用次数就指数级增长！")
print("这就是 O(2^n) 的恐怖之处。")

In [ ]:
# 方法二：记忆化递归（Memoization）
# Method 2: Memoization - cache results to avoid recomputation

call_count_memo = 0

def fib_memo(n, memo=None):
    """记忆化递归 - O(n) 时间复杂度"""
    global call_count_memo
    call_count_memo += 1
    
    if memo is None:
        memo = {}
    
    # 如果已经计算过，直接返回缓存结果
    if n in memo:
        return memo[n]
    
    if n <= 1:  # Base case
        return n
    
    # 计算并缓存
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


# 也可以使用 Python 内置装饰器
@lru_cache(maxsize=None)
def fib_lru(n):
    """使用 lru_cache 装饰器的记忆化"""
    if n <= 1:
        return n
    return fib_lru(n - 1) + fib_lru(n - 2)


print("=== 记忆化递归 ===")
for n in [5, 10, 20, 30, 50, 100]:
    call_count_memo = 0
    start = time.time()
    result = fib_memo(n)
    elapsed = time.time() - start
    print(f"fib({n:3d}) = {result:>25d} | 调用次数: {call_count_memo:>6d} | 耗时: {elapsed:.6f}秒")

print("\n对比朴素递归，调用次数大幅减少！")

In [ ]:
# 方法三：动态规划（自底向上）
# Method 3: Dynamic Programming (Bottom-up)

def fib_dp(n):
    """动态规划 - O(n) 时间, O(n) 空间"""
    if n <= 1:
        return n
    
    # 创建 dp 表
    dp = [0] * (n + 1)
    dp[0] = 0
    dp[1] = 1
    
    # 自底向上填表
    for i in range(2, n + 1):
        dp[i] = dp[i-1] + dp[i-2]
    
    return dp[n]


def fib_dp_optimized(n):
    """空间优化的动态规划 - O(n) 时间, O(1) 空间"""
    if n <= 1:
        return n
    
    # 只需要保存前两个值
    prev2, prev1 = 0, 1
    for i in range(2, n + 1):
        current = prev1 + prev2
        prev2 = prev1
        prev1 = current
    
    return prev1


print("=== 动态规划 ===")
for n in [5, 10, 20, 50, 100, 200]:
    start = time.time()
    result = fib_dp_optimized(n)
    elapsed = time.time() - start
    print(f"fib({n:3d}) = {result} (耗时: {elapsed:.6f}秒)")

print("\n动态规划从底部向上计算，完全不需要递归！")

In [ ]:
# 性能对比可视化 - Performance Comparison Visualization

def visualize_fib_comparison():
    """可视化三种 Fibonacci 实现的性能对比"""
    
    # 朴素递归只测试小数字
    naive_ns = list(range(5, 31, 5))
    naive_times = []
    for n in naive_ns:
        start = time.time()
        fib_naive(n)
        naive_times.append(time.time() - start)
    
    # 记忆化和动态规划可以测试大数字
    fast_ns = list(range(5, 201, 5))
    memo_times = []
    dp_times = []
    
    for n in fast_ns:
        start = time.time()
        fib_memo(n)
        memo_times.append(time.time() - start)
        
        start = time.time()
        fib_dp_optimized(n)
        dp_times.append(time.time() - start)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # 左图：朴素递归（注意时间尺度）
    ax1.plot(naive_ns, naive_times, 'ro-', linewidth=2, markersize=8, label='朴素递归 Naive')
    ax1.set_xlabel('n', fontsize=12)
    ax1.set_ylabel('时间（秒）', fontsize=12)
    ax1.set_title('朴素递归 O(2^n)\n指数级增长！', fontsize=13, fontweight='bold', color='red')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # 右图：记忆化 vs 动态规划
    ax2.plot(fast_ns, memo_times, 'b^-', linewidth=1.5, markersize=4, label='记忆化 Memo O(n)', alpha=0.7)
    ax2.plot(fast_ns, dp_times, 'gs-', linewidth=1.5, markersize=4, label='动态规划 DP O(n)', alpha=0.7)
    ax2.set_xlabel('n', fontsize=12)
    ax2.set_ylabel('时间（秒）', fontsize=12)
    ax2.set_title('记忆化 vs 动态规划\n线性增长', fontsize=13, fontweight='bold', color='green')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle('Fibonacci 三种实现方法性能对比', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

visualize_fib_comparison()

In [ ]:
# 递归树可视化 - Recursion Tree for Fibonacci

def visualize_fib_tree():
    """可视化 fib(5) 的递归树，展示重复计算"""
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_xlim(-1, 17)
    ax.set_ylim(-1, 11)
    ax.axis('off')
    ax.set_title('fib(5) 递归树 - 红色节点表示重复计算\nRecursion Tree - Red nodes show redundant computation',
                fontsize=14, fontweight='bold')
    
    # 手动定义树结构 (value, x, y, parent_x, parent_y, is_duplicate)
    nodes = [
        # (标签, x, y, 父x, 父y, 是否重复)
        ('f(5)', 8, 10, None, None, False),
        ('f(4)', 4, 8, 8, 10, False),
        ('f(3)', 12, 8, 8, 10, False),
        ('f(3)', 2, 6, 4, 8, True),
        ('f(2)', 6, 6, 4, 8, False),
        ('f(2)', 10, 6, 12, 8, True),
        ('f(1)', 14, 6, 12, 8, False),
        ('f(2)', 1, 4, 2, 6, True),
        ('f(1)', 3, 4, 2, 6, False),
        ('f(1)', 5, 4, 6, 6, False),
        ('f(0)', 7, 4, 6, 6, False),
        ('f(1)', 9, 4, 10, 6, False),
        ('f(0)', 11, 4, 10, 6, False),
        ('f(1)', 0, 2, 1, 4, False),
        ('f(0)', 2, 2, 1, 4, False),
    ]
    
    for label, x, y, px, py, is_dup in nodes:
        # 画连线
        if px is not None:
            ax.plot([px, x], [py - 0.4, y + 0.4], 'gray', linewidth=1)
        
        # 画节点
        color = '#FF6B6B' if is_dup else '#87CEEB'
        circle = plt.Circle((x, y), 0.4, facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(circle)
        ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # 图例
    ax.add_patch(plt.Circle((0.5, -0.3), 0.25, facecolor='#87CEEB', edgecolor='black'))
    ax.text(1.2, -0.3, '= 首次计算', va='center', fontsize=10)
    ax.add_patch(plt.Circle((4.5, -0.3), 0.25, facecolor='#FF6B6B', edgecolor='black'))
    ax.text(5.2, -0.3, '= 重复计算（浪费！）', va='center', fontsize=10, color='red')
    
    ax.text(10, -0.3, '记忆化(Memoization)可以消除所有重复计算！', fontsize=11, fontweight='bold', color='green')
    
    plt.tight_layout()
    plt.show()

visualize_fib_tree()

---
## 9. 递归 vs 迭代对比总结

| 特性 | 递归 Recursion | 迭代 Iteration |
|------|---------------|----------------|
| 代码简洁性 | 通常更简洁、更直观 | 可能需要更多代码 |
| 内存使用 | 需要调用栈空间 O(n) | 通常 O(1) |
| 执行速度 | 函数调用有开销 | 通常更快 |
| Stack Overflow 风险 | 递归深度过大会溢出 | 不会 |
| 适用场景 | 树/图遍历、分治、回溯 | 简单循环 |
| 考试重点 | 理解递归展开和基线条件 | 能将递归转为迭代 |

### 关键考点：
1. 识别递归函数的 base case 和 recursive case
2. 手动追踪（trace）递归函数的执行过程
3. 理解递归和分治法的关系
4. 知道哪些算法天然适合递归（树遍历、排序等）

In [ ]:
# 互动演示：递归求和 - Interactive Demo: Recursive Sum

def recursive_sum(lst, index=0, depth=0):
    """
    递归计算列表元素之和
    Recursively sum a list
    """
    indent = "  " * depth
    
    # 基线条件：已经遍历完所有元素
    if index == len(lst):
        print(f"{indent}基线条件: index={index} == len(lst)={len(lst)}, 返回 0")
        return 0
    
    print(f"{indent}recursive_sum(lst, {index}): 处理元素 lst[{index}] = {lst[index]}")
    rest_sum = recursive_sum(lst, index + 1, depth + 1)
    total = lst[index] + rest_sum
    print(f"{indent}返回: {lst[index]} + {rest_sum} = {total}")
    return total


print("=== 递归求和演示 ===")
test_list = [3, 7, 2, 5]
print(f"列表: {test_list}\n")
result = recursive_sum(test_list)
print(f"\n结果: {result}")
print(f"验证: sum({test_list}) = {sum(test_list)}")

In [ ]:
# 更多递归练习：字符串反转、回文检测
# More recursion examples: string reverse, palindrome check

def reverse_string(s):
    """递归反转字符串 Recursive string reversal"""
    # Base case: 空字符串或单字符
    if len(s) <= 1:
        return s
    # Recursive: 最后一个字符 + 反转剩余部分
    return s[-1] + reverse_string(s[:-1])


def is_palindrome(s):
    """递归检测回文 Recursive palindrome check"""
    # Base case
    if len(s) <= 1:
        return True
    # 首尾字符相同，且中间部分也是回文
    if s[0] == s[-1]:
        return is_palindrome(s[1:-1])
    return False


# 测试字符串反转
print("=== 递归字符串反转 ===")
test = "Hello"
print(f"原始: '{test}' → 反转: '{reverse_string(test)}'")

# 测试回文检测
print("\n=== 递归回文检测 ===")
for word in ["racecar", "level", "hello", "madam", "python"]:
    result = is_palindrome(word)
    print(f"  '{word}' → {'是回文 ✓' if result else '不是回文 ✗'}")

---
## 10. 分治策略的时间复杂度分析

### 主定理 Master Theorem（了解即可）

对于递归关系 $T(n) = aT(n/b) + O(n^d)$：

| 算法 | a | b | d | 复杂度 |
|------|---|---|---|--------|
| 二分查找 | 1 | 2 | 0 | O(log n) |
| 归并排序 | 2 | 2 | 1 | O(n log n) |
| 快速排序（平均） | 2 | 2 | 1 | O(n log n) |

### 考试重点：
- 二分查找：每次排除一半 → O(log n)
- 归并排序：分成两半 + 线性合并 → O(n log n)
- 快速排序：平均 O(n log n)，最坏 O(n^2)

In [ ]:
# 分治算法复杂度可视化

def visualize_complexity():
    """可视化不同分治算法的时间复杂度"""
    n = np.linspace(1, 100, 200)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    ax.plot(n, np.log2(n), 'g-', linewidth=2.5, label='O(log n) - 二分查找')
    ax.plot(n, n, 'b-', linewidth=2.5, label='O(n) - 线性查找')
    ax.plot(n, n * np.log2(n), 'orange', linewidth=2.5, label='O(n log n) - 归并/快排')
    ax.plot(n, n**2, 'r-', linewidth=2.5, label='O(n²) - 冒泡/选择排序')
    
    ax.set_xlabel('输入规模 n', fontsize=13)
    ax.set_ylabel('操作次数', fontsize=13)
    ax.set_title('分治算法 vs 普通算法的时间复杂度对比', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12, loc='upper left')
    ax.set_ylim(0, 3000)
    ax.grid(True, alpha=0.3)
    
    # 标注关键区域
    ax.annotate('分治法优势区域', xy=(60, 400), fontsize=12,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

visualize_complexity()

---
## 练习题 Practice Exercises

### 练习 1：递归基础
编写一个递归函数 `power(base, exp)`，计算 base 的 exp 次方。
- Base case: exp == 0 时返回 1
- Recursive case: base * power(base, exp - 1)

In [ ]:
# 练习 1：请在下方编写代码
# Exercise 1: Write your code below

def power(base, exp):
    """递归计算 base^exp"""
    # 请在这里写代码
    # Write your code here
    pass  # 删除这行，替换为你的代码


# 测试
# print(power(2, 10))  # 应该输出 1024
# print(power(3, 4))   # 应该输出 81
# print(power(5, 0))   # 应该输出 1

### 练习 2：归并排序
手动追踪 `merge_sort([5, 2, 8, 1, 9, 3])` 的完整执行过程。
写出每一步的分割和合并操作。

In [ ]:
# 练习 2：取消注释下面的代码来验证你的手动追踪
# Exercise 2: Uncomment to verify your manual trace

# merge_sort([5, 2, 8, 1, 9, 3])

### 练习 3：快速排序分区
给定数组 `[7, 2, 1, 6, 8, 5, 3, 4]`，以最后一个元素 4 为 pivot，
手动执行 Lomuto 分区过程，写出每一步数组的变化。

In [ ]:
# 练习 3：带追踪的分区函数
# Exercise 3: Partition with trace

def partition_trace(arr, low, high):
    """带详细追踪的 Lomuto 分区"""
    pivot = arr[high]
    print(f"Pivot = {pivot}")
    i = low - 1
    
    for j in range(low, high):
        print(f"  j={j}, arr[j]={arr[j]}: ", end="")
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
            print(f"交换 arr[{i}] 和 arr[{j}] → {arr}")
        else:
            print(f"不交换 → {arr}")
    
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    print(f"  最终: 交换 pivot 到位置 {i+1} → {arr}")
    return i + 1


test = [7, 2, 1, 6, 8, 5, 3, 4]
print(f"原始数组: {test}")
pivot_pos = partition_trace(test, 0, len(test) - 1)
print(f"Pivot 最终位置: {pivot_pos}")

### 练习 4：递归实战
编写递归函数 `count_digits(n)` 计算正整数 n 的位数。
例如：count_digits(12345) = 5

In [ ]:
# 练习 4：请在下方编写代码
# Exercise 4: Write your code below

def count_digits(n):
    """递归计算正整数的位数"""
    # 提示 Hint:
    # Base case: n < 10 时只有 1 位
    # Recursive case: 1 + count_digits(n // 10)
    pass  # 请替换


# 测试
# print(count_digits(12345))  # 应该输出 5
# print(count_digits(7))      # 应该输出 1
# print(count_digits(100))    # 应该输出 3

### 练习 5：思考题
1. 为什么快速排序最坏情况是 O(n^2)？什么情况下会出现？
2. 归并排序和快速排序，哪个是稳定排序（stable sort）？为什么？
3. 如何将朴素递归 Fibonacci 的时间复杂度从 O(2^n) 优化到 O(n)？
4. 递归都可以转换为迭代吗？如何转换？

---

### 参考答案要点：
1. 当 pivot 总是最大或最小元素时（如已排序数组选第一个），每次只减少一个元素，退化为 O(n^2)
2. 归并排序是稳定的，因为合并时相等元素保持原有顺序；快速排序不稳定
3. 使用记忆化（Memoization）缓存已计算结果，或使用动态规划自底向上计算
4. 理论上可以，使用显式栈（Stack）模拟递归调用栈

---
## 本节总结 Summary

| 概念 | 关键要点 |
|------|----------|
| 递归 Recursion | 函数调用自身，必须有 base case |
| 调用栈 Call Stack | 每次递归调用创建栈帧，LIFO 顺序返回 |
| 分治法 D&C | Divide → Conquer → Combine |
| 归并排序 Merge Sort | O(n log n)，稳定，需要 O(n) 额外空间 |
| 快速排序 Quick Sort | 平均 O(n log n)，最坏 O(n^2)，原地排序 |
| 二分查找 Binary Search | O(log n)，要求数组已排序 |
| 记忆化 Memoization | 缓存结果避免重复计算 |
| 动态规划 DP | 自底向上求解，避免递归开销 |